# fMRI · 01 · Train fMRI → CLIP (Experiment 1)
**Question:** does the model learn a useful CLIP representation from fMRI?

Training calls `src.training.train_clip` — the exact function used by `scripts/02_train_fmri_to_clip.py` (modality is config-driven). Set `RESUME=True` to continue from `last.pt`.

In [ ]:
# Run from the PROJECT ROOT so relative paths (configs/, data/, outputs/)
# resolve exactly like the scripts do.
import sys, os
_root = os.getcwd()
while _root != os.path.dirname(_root):
    if os.path.isdir(os.path.join(_root, 'src')) and os.path.isdir(os.path.join(_root, 'configs')):
        break
    _root = os.path.dirname(_root)
os.chdir(_root); sys.path.insert(0, _root)
print('project root:', os.getcwd())
import numpy as np
import matplotlib.pyplot as plt
from src.utils import load_config, get_experiment_paths, load_json
from src.training import train_clip

RESUME = False
cfg = load_config('configs/fMRI/exp01_fmri_to_clip.yaml')
# Precompute CLIP features first: python scripts/01_precompute_clip.py --config configs/fMRI/exp01_fmri_to_clip.yaml

In [ ]:
# Train (a high epoch ceiling; early stopping picks best.pt):
cfg['training']['epochs'] = 100
result = train_clip(cfg, resume=('auto' if RESUME else None))
result['metrics']

## Loss and retrieval curves

In [ ]:
%matplotlib inline

In [ ]:
import pandas as pd
paths = get_experiment_paths(cfg, ensure=False)
log = pd.read_csv(paths.train_log)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(log.epoch, log.train_total, label='train')
ax[0].plot(log.epoch, log.val_loss, label='val'); ax[0].legend(); ax[0].set_title('loss')
for c in ['val_top1','val_top5','val_top10']:
    ax[1].plot(log.epoch, log[c], label=c)
ax[1].legend(); ax[1].set_title('val retrieval'); plt.tight_layout(); plt.show()

In [ ]:
print('TEST metrics:'); load_json(paths.metrics / 'test_metrics.json')

**Takeaway:** rising Top-k on val, plus Top-k clearly above chance (`k / n_candidates`), indicate the fMRI carries CLIP-decodable information. The decisive control comes next (notebook 02).